In [ ]:
# ============================================================
# NOTEBOOK : Nettoyage et Pré-traitement — GPS Spoofing
# Dataset  : Signal-level GPS attack captures
# ============================================================

# ────────────────────────────────────────
# 1. IMPORTATION DES LIBRAIRIES
# ────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
import os

# ────────────────────────────────────────
# 2. CHARGEMENT DES DATASETS
# ────────────────────────────────────────
PATH = r"C:\Drone_Attack_Similarity_Project\DATASET\GPS_Spoofing"

df1 = pd.read_csv(os.path.join(PATH, "GPS_Authentic.csv"))
df2 = pd.read_csv(os.path.join(PATH, "GPS_Simplified.csv"))
df3 = pd.read_csv(os.path.join(PATH, "GPS_Full.csv"))

df1_original = df1.copy()
df2_original = df2.copy()
df3_original = df3.copy()

print("  Fichiers chargés")
print(f"   df1 — GPS_Authentic  : {df1.shape[0]:,} lignes × {df1.shape[1]} colonnes")
print(f"   df2 — GPS_Simplified : {df2.shape[0]:,} lignes × {df2.shape[1]} colonnes")
print(f"   df3 — GPS_Full       : {df3.shape[0]:,} lignes × {df3.shape[1]} colonnes")

# ────────────────────────────────────────
# 3. SUPPRESSION DES DOUBLONS
# ────────────────────────────────────────
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    nb = df.duplicated().sum()
    if nb > 0:
        df.drop_duplicates(inplace=True)
        print(f" {name} — {nb} doublons supprimés — nouveau shape : {df.shape}")
    else:
        print(f" {name} — Aucun doublon détecté")



# ────────────────────────────────────────
# 4. GESTION DES VALEURS MANQUANTES
# ────────────────────────────────────────
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    total = df.isnull().sum().sum()
    print(f"\n {name} — Total valeurs manquantes : {total}")

    if total > 0:
        num_cols = df.select_dtypes(include=[np.number]).columns
        cat_cols = df.select_dtypes(include=['object']).columns

        for col in num_cols:
            if df[col].isnull().sum() > 0:
                median_val = df[col].median()
                df[col].fillna(median_val, inplace=True)
                print(f"  {col} — rempli par médiane ({median_val:.4f})")

        for col in cat_cols:
            if df[col].isnull().sum() > 0:
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                print(f"{col} — rempli par mode ({mode_val})")
    else:
        print(f" Aucune valeur manquante")


# ────────────────────────────────────────
# 5. SUPPRESSION DES COLONNES INUTILES
# ────────────────────────────────────────
# Les colonnes Unnamed viennent de la conversion xlsx → csv
for name, df in [("df1", df1), ("df2", df2), ("df3", df3)]:
    unnamed_cols = [c for c in df.columns if 'Unnamed' in str(c)]
    zero_var_cols = [c for c in df.columns
                     if df[c].nunique() == 1 and c != 'Output']

    cols_to_drop = list(set(unnamed_cols + zero_var_cols))

    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
        print(f" {name} — {len(cols_to_drop)} colonnes supprimées : {cols_to_drop[:5]}...")
    else:
        print(f" {name} — Aucune colonne inutile")

    print(f"   Shape après suppression : {df.shape}")


# ────────────────────────────────────────
# 6. TRAITEMENT DES OUTLIERS (IQR) — df2
# ────────────────────────────────────────
# On applique sur df2 (version simplifiée — la plus utilisée)
num_cols_df2 = [c for c in df2.select_dtypes(include=[np.number]).columns
                if c != 'Output']

print(" Traitement des outliers — df2 (écrêtage IQR) :\n")

nb_total_outliers = 0
for col in num_cols_df2:
    Q1  = df2[col].quantile(0.25)
    Q3  = df2[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    nb = ((df2[col] < lower) | (df2[col] > upper)).sum()
    nb_total_outliers += nb

    if nb > 0:
        df2[col] = df2[col].clip(lower=lower, upper=upper)
        print(f" {col} — {nb} outliers écrêtés")

print(f"\n Total outliers traités (df2) : {nb_total_outliers}")


# Même traitement pour df3
num_cols_df3 = [c for c in df3.select_dtypes(include=[np.number]).columns
                if c != 'Output']

print(" Traitement des outliers — df3 (écrêtage IQR) :\n")

nb_total_outliers_df3 = 0
for col in num_cols_df3:
    Q1  = df3[col].quantile(0.25)
    Q3  = df3[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    nb = ((df3[col] < lower) | (df3[col] > upper)).sum()
    nb_total_outliers_df3 += nb

    if nb > 0:
        df3[col] = df3[col].clip(lower=lower, upper=upper)
        print(f" {col} — {nb} outliers écrêtés")

print(f"\n Total outliers traités (df3) : {nb_total_outliers_df3}")


# ────────────────────────────────────────
# 7. NORMALISATION DES FEATURES (StandardScaler)
# ────────────────────────────────────────
scaler = StandardScaler()

# df2
num_cols_df2 = [c for c in df2.select_dtypes(include=[np.number]).columns
                if c != 'Output']
df2[num_cols_df2] = scaler.fit_transform(df2[num_cols_df2])
print(f" df2 — Normalisation appliquée sur {len(num_cols_df2)} features")

# df3
scaler2 = StandardScaler()
num_cols_df3 = [c for c in df3.select_dtypes(include=[np.number]).columns
                if c != 'Output']
df3[num_cols_df3] = scaler2.fit_transform(df3[num_cols_df3])
print(f" df3 — Normalisation appliquée sur {len(num_cols_df3)} features")

# df1 (pas de label)
scaler3 = StandardScaler()
num_cols_df1 = df1.select_dtypes(include=[np.number]).columns.tolist()
df1[num_cols_df1] = scaler3.fit_transform(df1[num_cols_df1])
print(f" df1 — Normalisation appliquée sur {len(num_cols_df1)} features")

print(f"\n   Vérification df2 après normalisation :")
print(df2[num_cols_df2].describe().round(2).T[['mean', 'std', 'min', 'max']])


# ────────────────────────────────────────
# 8. ENCODAGE DU LABEL (Output)
# ────────────────────────────────────────
# df1 n'a pas de label
# df2 et df3 ont une colonne 'Output'

le = LabelEncoder()

# df2
print(" df2 — Classes avant encodage :")
print(df2['Output'].value_counts())
df2['label_encoded'] = le.fit_transform(df2['Output'].astype(str))
print(f"\n df2 — Encodage appliqué :")
mapping = dict(zip(le.classes_, le.transform(le.classes_)))
for classe, code in mapping.items():
    print(f"   {classe} → {code}")

le2 = LabelEncoder()

# df3
print(" df3 — Classes avant encodage :")
print(df3['Output'].value_counts())
df3['label_encoded'] = le2.fit_transform(df3['Output'].astype(str))
print(f"\n df3 — Encodage appliqué :")
mapping2 = dict(zip(le2.classes_, le2.transform(le2.classes_)))
for classe, code in mapping2.items():
    print(f"   {classe} → {code}")

# ────────────────────────────────────────
# 9. VÉRIFICATION FINALE
# ────────────────────────────────────────
print(" Vérification finale :\n")

for name, df_orig, df_clean in [
    ("df1", df1_original, df1),
    ("df2", df2_original, df2),
    ("df3", df3_original, df3)
]:
    print(f"   {name} :")
    print(f"      Shape original  : {df_orig.shape}")
    print(f"      Shape nettoyé   : {df_clean.shape}")
    print(f"      Valeurs manquantes : {df_clean.isnull().sum().sum()}")
    print(f"      Doublons restants  : {df_clean.duplicated().sum()}")
    print()


# ────────────────────────────────────────
# 10. SAUVEGARDE DES DATASETS NETTOYÉS
# ────────────────────────────────────────
df1.to_csv(os.path.join(PATH, "C:\Drone_Attack_Similarity_Project\Rapport\tables\GPS_Authentic_clean.csv"),   index=False)
df2.to_csv(os.path.join(PATH, "C:\Drone_Attack_Similarity_Project\Rapport\tables\GPS_Simplified_clean.csv"),  index=False)
df3.to_csv(os.path.join(PATH, "C:\Drone_Attack_Similarity_Project\Rapport\tables\GPS_Full_clean.csv"),        index=False)

print(" Datasets nettoyés sauvegardés :")
print(f"   GPS_Authentic_clean.csv  : {df1.shape}")
print(f"   GPS_Simplified_clean.csv : {df2.shape}")
print(f"   GPS_Full_clean.csv       : {df3.shape}")